In [ ]:
#Install libraries
!pip install pulp -q

# Problema de Optimización: Producción de Muebles

Una fábrica produce dos tipos de muebles: **sillas** y **mesas**.

- Cada silla genera una utilidad de **30**.
- Cada mesa genera una utilidad de **50**.

## Restricciones

### 1. Tiempo de producción
- Cada silla requiere **2 horas** de trabajo.
- Cada mesa requiere **5 horas** de trabajo.
- La fábrica dispone de un máximo de **100 horas** por día.

### 2. Materiales
- Cada silla requiere **1 unidad de madera**.
- Cada mesa requiere **3 unidades de madera**.
- La fábrica dispone de un máximo de **60 unidades de madera** por día.

### 3. Demanda
- La demanda diaria máxima es de **20 sillas**.
- La demanda diaria máxima es de **10 mesas**.

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL - CENTRO DE DATOS
# EJERCICIO #1 - facil
# ============================================================
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus, value

# ============================================================
# DATOS DEL PROBLEMA
# ============================================================

productos = {
    "X1": {
        "nombre": "Sillas",
        "descripcion": "número de sillas que debe producir la fábrica diariamente",
        "utilidad": 30,
        "tiempo": 2,
        "material": 1,
        "demanda": 20
    },
    "X2": {
        "nombre": "Mesas",
        "descripcion": "número de mesas que debe producir la fábrica diariamente",
        "utilidad": 50,
        "tiempo": 5,
        "material": 3,
        "demanda": 10
    }
}

recursos = {
    "Tiempo": {
        "coeficiente": "tiempo",
        "disponible": 100
    },
    "Material": {
        "coeficiente": "material",
        "disponible": 60
    }
}

# ============================================================
# MODELO
# ============================================================

modelo = LpProblem("Maximizacion_Utilidad", LpMaximize)

variables = {
    codigo: LpVariable(codigo, lowBound=0, cat="Integer")
    for codigo in productos
}

modelo += sum(
    datos["utilidad"] * variables[codigo]
    for codigo, datos in productos.items()
)

for nombre_recurso, recurso in recursos.items():
    modelo += sum(
        productos[codigo][recurso["coeficiente"]] * variables[codigo]
        for codigo in productos
    ) <= recurso["disponible"]

for codigo, datos in productos.items():
    modelo += variables[codigo] <= datos["demanda"]

modelo.solve()

# ============================================================
# SALIDA FORMATEADA
# ============================================================

print("1. VARIABLES DE DECISIÓN")
for codigo, datos in productos.items():
    print(f"{codigo} = {datos['descripcion']}")
print()

print("2. FUNCIÓN OBJETIVO")
funcion_objetivo = " + ".join(
    f"{datos['utilidad']}{codigo}"
    for codigo, datos in productos.items()
)
print(f"Max Z = {funcion_objetivo}")
print()

print("3. RESTRICCIONES")

for nombre_recurso, recurso in recursos.items():
    expresion = " + ".join(
        f"{productos[codigo][recurso['coeficiente']]}{codigo}"
        for codigo in productos
    )
    print(
        f"{expresion} <= {recurso['disponible']}   "
        f"-> Restricción de {nombre_recurso.lower()}"
    )

for codigo, datos in productos.items():
    print(f"{codigo} <= {datos['demanda']}           -> Demanda máxima de {datos['nombre'].lower()}")

variables_no_negativas = ", ".join(productos.keys())
print(f"{variables_no_negativas} >= 0")
print()

print("4. SOLUCIÓN DEL MODELO")
print("Estado de la solución:", LpStatus[modelo.status])
print()

print("Valores óptimos encontrados:")
for codigo, datos in productos.items():
    print(f"{codigo} ({datos['nombre']}) = {int(variables[codigo].varValue)}")
print()

print("Utilidad máxima:")
print("Z =", int(value(modelo.objective)))
print("\n" + "=" * 60 + "\n")

print("5. VERIFICACIÓN DE RESTRICCIONES")

for nombre_recurso, recurso in recursos.items():
    partes = []
    total_usado = 0

    for codigo in productos:
        coeficiente = productos[codigo][recurso["coeficiente"]]
        cantidad = int(variables[codigo].varValue)
        total_usado += coeficiente * cantidad
        partes.append(f"{coeficiente}({cantidad})")

    expresion = " + ".join(partes)

    print(
        f"{nombre_recurso} usado: {expresion} = "
        f"{total_usado} <= {recurso['disponible']}"
    )

for codigo, datos in productos.items():
    cantidad = int(variables[codigo].varValue)
    print(f"Demanda {datos['nombre'].lower()}: {cantidad} <= {datos['demanda']}")

print()

print("7. RESPUESTA FINAL")
print(
    f"La fábrica debe producir {int(variables['X1'].varValue)} sillas "
    f"y {int(variables['X2'].varValue)} mesas por día."
)
print(f"La utilidad máxima que obtiene es de {int(value(modelo.objective))}.")

1. VARIABLES DE DECISIÓN
X1 = número de sillas que debe producir la fábrica diariamente
X2 = número de mesas que debe producir la fábrica diariamente

2. FUNCIÓN OBJETIVO
Max Z = 30X1 + 50X2

3. RESTRICCIONES
2X1 + 5X2 <= 100   -> Restricción de tiempo
1X1 + 3X2 <= 60   -> Restricción de material
X1 <= 20           -> Demanda máxima de sillas
X2 <= 10           -> Demanda máxima de mesas
X1, X2 >= 0

4. SOLUCIÓN DEL MODELO
Estado de la solución: Optimal

Valores óptimos encontrados:
X1 (Sillas) = 20
X2 (Mesas) = 10

Utilidad máxima:
Z = 1100


5. VERIFICACIÓN DE RESTRICCIONES
Tiempo usado: 2(20) + 5(10) = 90 <= 100
Material usado: 1(20) + 3(10) = 50 <= 60
Demanda sillas: 20 <= 20
Demanda mesas: 10 <= 10

7. RESPUESTA FINAL
La fábrica debe producir 20 sillas y 10 mesas por día.
La utilidad máxima que obtiene es de 1100.


# Problema de Asignación de Aplicaciones en Servidores

Un centro de datos tiene tres servidores (**S1, S2, S3**) que deben ejecutar cuatro aplicaciones (**A1, A2, A3, A4**).

Cada servidor tiene una **capacidad de procesamiento limitada (en GHz)** y un **consumo de energía diferente**.  
Cada aplicación requiere un mínimo de capacidad de procesamiento para funcionar correctamente.

## Objetivo

Asignar las aplicaciones a los servidores de manera que se **minimice el consumo total de energía**, sin exceder la capacidad de los servidores.

## Datos

### 1. Capacidad de los servidores (en GHz)
- **S1:** 100 GHz  
- **S2:** 150 GHz  
- **S3:** 200 GHz  

### 2. Consumo de energía por GHz
- **S1:** 0.5 W/GHz  
- **S2:** 0.4 W/GHz  
- **S3:** 0.3 W/GHz  

### 3. Requerimientos de las aplicaciones (en GHz)
- **A1:** 50 GHz  
- **A2:** 60 GHz  
- **A3:** 80 GHz  
- **A4:** 70 GHz  

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL - CENTRO DE DATOS
# EJERCICIO #2 - facil
# ============================================================
from pulp import LpMinimize, LpProblem, LpVariable, LpStatus, value

servidores = {
    "S1": {"capacidad": 100, "consumo": 0.5},
    "S2": {"capacidad": 150, "consumo": 0.4},
    "S3": {"capacidad": 200, "consumo": 0.3}
}

aplicaciones = {
    "A1": {"requerimiento": 50},
    "A2": {"requerimiento": 60},
    "A3": {"requerimiento": 80},
    "A4": {"requerimiento": 70}
}

modelo = LpProblem("Centro_de_Datos", LpMinimize)

# Variables de decisión xij
x = {}

for i, servidor in enumerate(servidores, start=1):
    for j, aplicacion in enumerate(aplicaciones, start=1):
        nombre_variable = f"x{i}{j}"
        x[servidor, aplicacion] = LpVariable(nombre_variable, lowBound=0)

# Función objetivo
modelo += sum(
    servidores[servidor]["consumo"] * x[servidor, aplicacion]
    for servidor in servidores
    for aplicacion in aplicaciones
), "Consumo_Total_Energia"

# Restricciones de capacidad de servidores
for servidor in servidores:
    modelo += sum(
        x[servidor, aplicacion]
        for aplicacion in aplicaciones
    ) <= servidores[servidor]["capacidad"], f"Capacidad_{servidor}"

# Restricciones de requerimiento de aplicaciones
for aplicacion in aplicaciones:
    modelo += sum(
        x[servidor, aplicacion]
        for servidor in servidores
    ) == aplicaciones[aplicacion]["requerimiento"], f"Requerimiento_{aplicacion}"

# Resolver modelo
modelo.solve()

# ============================================================
# 1) Datos
# ============================================================
print("Datos:")

print("Capacidad de los servidores:")
for servidor, datos in servidores.items():
    print(f"{servidor} = {datos['capacidad']} GHz")

print()

print("Consumo de energía por GHz:")
for servidor, datos in servidores.items():
    print(f"{servidor} = {datos['consumo']} W/GHz")

print()

print("Requerimientos de las aplicaciones:")
for aplicacion, datos in aplicaciones.items():
    print(f"{aplicacion} = {datos['requerimiento']} GHz")

print("\n" + "=" * 70 + "\n")

# ============================================================
# 2) VARIABLES DE DECISIÓN
# ============================================================
print("2) VARIABLES DE DECISIÓN")

for i, servidor in enumerate(servidores, start=1):
    for j, aplicacion in enumerate(aplicaciones, start=1):
        print(
            f"x{i}{j} = GHz del servidor {servidor} "
            f"asignados a la aplicación {aplicacion}"
        )
    print()

# ============================================================
# 3) FUNCIÓN OBJETIVO
# ============================================================
print("3) FUNCIÓN OBJETIVO")

lineas_objetivo = []

for i, servidor in enumerate(servidores, start=1):
    terminos = []

    for j, aplicacion in enumerate(aplicaciones, start=1):
        consumo = servidores[servidor]["consumo"]
        terminos.append(f"{consumo}x{i}{j}")

    lineas_objetivo.append(" + ".join(terminos))

print(f"Min Z = {lineas_objetivo[0]}")
for linea in lineas_objetivo[1:]:
    print(f"      + {linea}")

print()

# ============================================================
# 4) RESTRICCIONES
# ============================================================
print("4) RESTRICCIONES")

print("Capacidad de los servidores:")
for i, servidor in enumerate(servidores, start=1):
    expresion = " + ".join(
        f"x{i}{j}"
        for j, aplicacion in enumerate(aplicaciones, start=1)
    )
    capacidad = servidores[servidor]["capacidad"]
    print(f"{expresion} <= {capacidad}")

print()

print("Requerimiento de las aplicaciones:")
for j, aplicacion in enumerate(aplicaciones, start=1):
    expresion = " + ".join(
        f"x{i}{j}"
        for i, servidor in enumerate(servidores, start=1)
    )
    requerimiento = aplicaciones[aplicacion]["requerimiento"]
    print(f"{expresion} = {requerimiento}")

print()

print("No negatividad:")
print("xij >= 0")
print()

# ============================================================
# 5) SOLUCIÓN
# ============================================================
print("5) SOLUCIÓN")
print("Estado de la solución:", LpStatus[modelo.status])
print()

print("Valores óptimos de las variables:")

for i, servidor in enumerate(servidores, start=1):
    for j, aplicacion in enumerate(aplicaciones, start=1):
        print(f"x{i}{j} = {x[servidor, aplicacion].varValue}")
    print()

print("Consumo mínimo total de energía:")
print(f"Z = {value(modelo.objective)} W")
print()

print("Uso total de cada servidor:")

for servidor in servidores:
    uso = sum(
        x[servidor, aplicacion].varValue
        for aplicacion in aplicaciones
    )
    capacidad = servidores[servidor]["capacidad"]
    print(f"{servidor} = {uso} GHz de {capacidad} GHz")

print()

print("Procesamiento total asignado a cada aplicación:")

for aplicacion in aplicaciones:
    asignado = sum(
        x[servidor, aplicacion].varValue
        for servidor in servidores
    )
    print(f"{aplicacion} = {asignado} GHz")

print()

print("Interpretación de la solución:")
print("La solución óptima asigna primero la mayor cantidad posible al servidor S3")
print("porque tiene el menor consumo por GHz, luego al servidor S2 y por último a S1 si es necesario.")

Datos:
Capacidad de los servidores:
S1 = 100 GHz
S2 = 150 GHz
S3 = 200 GHz

Consumo de energía por GHz:
S1 = 0.5 W/GHz
S2 = 0.4 W/GHz
S3 = 0.3 W/GHz

Requerimientos de las aplicaciones:
A1 = 50 GHz
A2 = 60 GHz
A3 = 80 GHz
A4 = 70 GHz


2) VARIABLES DE DECISIÓN
x11 = GHz del servidor S1 asignados a la aplicación A1
x12 = GHz del servidor S1 asignados a la aplicación A2
x13 = GHz del servidor S1 asignados a la aplicación A3
x14 = GHz del servidor S1 asignados a la aplicación A4

x21 = GHz del servidor S2 asignados a la aplicación A1
x22 = GHz del servidor S2 asignados a la aplicación A2
x23 = GHz del servidor S2 asignados a la aplicación A3
x24 = GHz del servidor S2 asignados a la aplicación A4

x31 = GHz del servidor S3 asignados a la aplicación A1
x32 = GHz del servidor S3 asignados a la aplicación A2
x33 = GHz del servidor S3 asignados a la aplicación A3
x34 = GHz del servidor S3 asignados a la aplicación A4

3) FUNCIÓN OBJETIVO
Min Z = 0.5x11 + 0.5x12 + 0.5x13 + 0.5x14
      + 0.4x21

Una empresa debe abastecer a tres clientes (Cliente 1, Cliente 2 y Cliente 3) utilizando dos posibles bodegas (A y B). Cada cliente tiene una demanda semanal que debe ser satisfecha completamente: el Cliente 1 requiere 80 unidades, el Cliente 2 requiere 70 unidades y el Cliente 3 requiere 60 unidades.
Las bodegas tienen una capacidad máxima de envío semanal. La Bodega A puede enviar hasta 150 unidades y la Bodega B hasta 120 unidades. Además, abrir una bodega genera un costo fijo: abrir la Bodega A cuesta 500 y abrir la Bodega B cuesta 400.
Existen costos de transporte por unidad desde cada bodega hacia cada cliente. Desde la Bodega A, el costo por unidad es de 4 hacia el Cliente 1, 6 hacia el Cliente 2 y 9 hacia el Cliente 3. Desde la Bodega B, el costo por unidad es de 5 hacia el Cliente 1, 4 hacia el Cliente 2 y 7 hacia el Cliente 3.
La empresa debe decidir qué bodegas abrir y cuántas unidades enviar desde cada bodega a cada cliente, de tal manera que se satisfaga completamente la demanda de los clientes y no se exceda la capacidad de las bodegas.
Formular el modelo que permita determinar la cantidad a enviar desde cada bodega a cada cliente y la apertura de las bodegas, con el fin de minimizar los costos totales, considerando los costos fijos de apertura y los costos de transporte.

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL ENTERA MIXTA (MLP)
# Apertura de bodegas y transporte
# Ejercicio #3 - medio
# ============================================================
from pulp import LpMinimize, LpProblem, LpVariable, LpStatus, value, LpBinary

# ============================================================
# 1) DATOS DEL PROBLEMA
# ============================================================

bodegas = {
    "A": {
        "capacidad": 150,
        "costo_fijo": 500
    },
    "B": {
        "capacidad": 120,
        "costo_fijo": 400
    }
}

clientes = {
    "1": {"demanda": 80},
    "2": {"demanda": 70},
    "3": {"demanda": 60}
}

costos_transporte = {
    ("A", "1"): 4,
    ("A", "2"): 6,
    ("A", "3"): 9,
    ("B", "1"): 5,
    ("B", "2"): 4,
    ("B", "3"): 7
}

# ============================================================
# 2) CREACIÓN DEL MODELO
# ============================================================

modelo = LpProblem("Apertura_Bodegas_y_Transporte", LpMinimize)

# Variables de envío xij
x = {
    (bodega, cliente): LpVariable(f"x{bodega}{cliente}", lowBound=0)
    for bodega in bodegas
    for cliente in clientes
}

# Variables binarias de apertura yi
y = {
    bodega: LpVariable(f"y{bodega}", cat=LpBinary)
    for bodega in bodegas
}

# Función objetivo
modelo += (
    sum(bodegas[bodega]["costo_fijo"] * y[bodega] for bodega in bodegas)
    +
    sum(costos_transporte[bodega, cliente] * x[bodega, cliente]
        for bodega in bodegas
        for cliente in clientes)
), "Costo_Total"

# Restricciones de demanda
for cliente in clientes:
    modelo += sum(
        x[bodega, cliente]
        for bodega in bodegas
    ) == clientes[cliente]["demanda"], f"Demanda_Cliente_{cliente}"

# Restricciones de capacidad ligadas a apertura
for bodega in bodegas:
    modelo += sum(
        x[bodega, cliente]
        for cliente in clientes
    ) <= bodegas[bodega]["capacidad"] * y[bodega], f"Capacidad_Bodega_{bodega}"

# Resolver modelo
modelo.solve()

# ============================================================
# 3) SALIDA FORMATEADA
# ============================================================

print("1) VARIABLES DE DECISIÓN")
print("xij = unidades enviadas desde la bodega i al cliente j")

for bodega in bodegas:
    print(f"y{bodega} = 1 si se abre la Bodega {bodega}, 0 en caso contrario")

print()

for bodega in bodegas:
    for cliente in clientes:
        print(f"x{bodega}{cliente} = unidades enviadas desde {bodega} al Cliente {cliente}")

print()

# ============================================================
# 4) FUNCIÓN OBJETIVO
# ============================================================

print("2) FUNCIÓN OBJETIVO")

terminos_fijos = [
    f"{bodegas[bodega]['costo_fijo']}y{bodega}"
    for bodega in bodegas
]

terminos_variables = [
    f"{costos_transporte[bodega, cliente]}x{bodega}{cliente}"
    for bodega in bodegas
    for cliente in clientes
]

print("Min Z = " + " + ".join(terminos_fijos + terminos_variables))
print()

# ============================================================
# 5) RESTRICCIONES
# ============================================================

print("3) RESTRICCIONES")

print("Demanda de los clientes:")
for cliente in clientes:
    expresion = " + ".join(
        f"x{bodega}{cliente}"
        for bodega in bodegas
    )
    print(f"{expresion} = {clientes[cliente]['demanda']}")

print()

print("Capacidad de las bodegas:")
for bodega in bodegas:
    expresion = " + ".join(
        f"x{bodega}{cliente}"
        for cliente in clientes
    )
    print(f"{expresion} <= {bodegas[bodega]['capacidad']}y{bodega}")

print()

print("No negatividad:")
print("xij >= 0")
print()

print("Variables binarias:")
variables_binarias = ", ".join(f"y{bodega}" for bodega in bodegas)
print(f"{variables_binarias} = 0 o 1")
print()

# ============================================================
# 6) SOLUCIÓN
# ============================================================

print("4) SOLUCIÓN")
print("Estado de la solución:", LpStatus[modelo.status])
print()

print("Valores óptimos de las variables:")

for bodega in bodegas:
    for cliente in clientes:
        print(f"x{bodega}{cliente} = {x[bodega, cliente].varValue}")
    print()

for bodega in bodegas:
    print(f"y{bodega} = {y[bodega].varValue}")

print()

print("Costo mínimo total:")
print(f"Z = {value(modelo.objective)}")
print()

print("Uso total de cada bodega:")
for bodega in bodegas:
    uso = sum(
        x[bodega, cliente].varValue
        for cliente in clientes
    )
    capacidad = bodegas[bodega]["capacidad"]
    print(f"Bodega {bodega} = {uso} unidades de {capacidad}")

print()

print("Demanda satisfecha de cada cliente:")
for cliente in clientes:
    demanda_satisfecha = sum(
        x[bodega, cliente].varValue
        for bodega in bodegas
    )
    print(f"Cliente {cliente} = {demanda_satisfecha} unidades")

print()

print("Interpretación de la solución:")
for bodega in bodegas:
    if y[bodega].varValue == 1:
        print(f"Se abre la Bodega {bodega}")
    else:
        print(f"No se abre la Bodega {bodega}")

print()

for bodega in bodegas:
    for cliente in clientes:
        print(
            f"Enviar {x[bodega, cliente].varValue} unidades "
            f"desde {bodega} al Cliente {cliente}"
        )

1) VARIABLES DE DECISIÓN
xij = unidades enviadas desde la bodega i al cliente j
yA = 1 si se abre la Bodega A, 0 en caso contrario
yB = 1 si se abre la Bodega B, 0 en caso contrario

xA1 = unidades enviadas desde A al Cliente 1
xA2 = unidades enviadas desde A al Cliente 2
xA3 = unidades enviadas desde A al Cliente 3
xB1 = unidades enviadas desde B al Cliente 1
xB2 = unidades enviadas desde B al Cliente 2
xB3 = unidades enviadas desde B al Cliente 3

2) FUNCIÓN OBJETIVO
Min Z = 500yA + 400yB + 4xA1 + 6xA2 + 9xA3 + 5xB1 + 4xB2 + 7xB3

3) RESTRICCIONES
Demanda de los clientes:
xA1 + xB1 = 80
xA2 + xB2 = 70
xA3 + xB3 = 60

Capacidad de las bodegas:
xA1 + xA2 + xA3 <= 150yA
xB1 + xB2 + xB3 <= 120yB

No negatividad:
xij >= 0

Variables binarias:
yA, yB = 0 o 1

4) SOLUCIÓN
Estado de la solución: Optimal

Valores óptimos de las variables:
xA1 = 80.0
xA2 = 0.0
xA3 = 10.0

xB1 = 0.0
xB2 = 70.0
xB3 = 50.0

yA = 1.0
yB = 1.0

Costo mínimo total:
Z = 1940.0

Uso total de cada bodega:
Bodega A = 90

Una empresa manufacturera produce dos productos: A y B. Para ello puede operar hasta dos plantas, Planta 1 y Planta 2. Abrir una planta implica un costo fijo semanal de 600 para la Planta 1 y de 500 para la Planta 2.
La empresa cuenta con recursos limitados para la producción. Dispone de un máximo de 200 horas de mano de obra y 150 kg de materia prima por semana. Cada unidad del Producto A requiere 3 horas de mano de obra y 2 kg de materia prima, mientras que cada unidad del Producto B requiere 2 horas de mano de obra y 3 kg de materia prima.
Las plantas tienen una capacidad máxima de producción total. La Planta 1 puede producir hasta 80 unidades en total y la Planta 2 hasta 100 unidades en total. Cada planta puede producir ambos productos, pero únicamente si ha sido abierta.
La utilidad generada por cada unidad producida es de 30 para el Producto A y de 40 para el Producto B. Además, la empresa debe cumplir con una demanda mínima semanal de 40 unidades del Producto A y 30 unidades del Producto B.
La empresa debe decidir qué plantas abrir y cuántas unidades de cada producto producir en cada planta, con el fin de maximizar la ganancia total, considerando las utilidades obtenidas por la producción y los costos fijos de apertura de las plantas.
Formular el modelo que permita determinar la apertura de las plantas y la cantidad a producir de cada producto en cada una de ellas para maximizar la ganancia total, sujeto a las restricciones de mano de obra, materia prima, capacidad de producción y demanda mínima.

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL ENTERA MIXTA (MLP)
# Producción con apertura de plantas
# Ejercicio #4 - medio
# ============================================================
from pulp import LpMaximize, LpProblem, LpVariable, LpStatus, value, LpBinary

# ============================================================
# 1) DATOS DEL PROBLEMA
# ============================================================

plantas = {
    "1": {
        "nombre": "Planta 1",
        "capacidad": 80,
        "costo_fijo": 600
    },
    "2": {
        "nombre": "Planta 2",
        "capacidad": 100,
        "costo_fijo": 500
    }
}

productos = {
    "A": {
        "utilidad": 30,
        "demanda_minima": 40,
        "recursos": {
            "Mano de obra": 3,
            "Materia prima": 2
        }
    },
    "B": {
        "utilidad": 40,
        "demanda_minima": 30,
        "recursos": {
            "Mano de obra": 2,
            "Materia prima": 3
        }
    }
}

recursos = {
    "Mano de obra": {
        "unidad": "horas",
        "disponible": 200
    },
    "Materia prima": {
        "unidad": "kg",
        "disponible": 150
    }
}

# ============================================================
# 2) CREACIÓN DEL MODELO
# ============================================================

modelo = LpProblem("Produccion_Plantas", LpMaximize)

x = {
    (planta, producto): LpVariable(f"x{planta}{producto}", lowBound=0)
    for planta in plantas
    for producto in productos
}

y = {
    planta: LpVariable(f"y{planta}", cat=LpBinary)
    for planta in plantas
}

# Función objetivo
modelo += (
    sum(
        productos[producto]["utilidad"] * x[planta, producto]
        for planta in plantas
        for producto in productos
    )
    -
    sum(
        plantas[planta]["costo_fijo"] * y[planta]
        for planta in plantas
    )
), "Ganancia_Total"

# Restricciones de recursos
for recurso, datos_recurso in recursos.items():
    modelo += sum(
        productos[producto]["recursos"][recurso] * x[planta, producto]
        for planta in plantas
        for producto in productos
    ) <= datos_recurso["disponible"], f"Restriccion_{recurso.replace(' ', '_')}"

# Restricciones de capacidad por planta
for planta in plantas:
    modelo += sum(
        x[planta, producto]
        for producto in productos
    ) <= plantas[planta]["capacidad"] * y[planta], f"Capacidad_Planta_{planta}"

# Restricciones de demanda mínima
for producto in productos:
    modelo += sum(
        x[planta, producto]
        for planta in plantas
    ) >= productos[producto]["demanda_minima"], f"Demanda_Minima_{producto}"

# Resolver
modelo.solve()

# ============================================================
# 3) VARIABLES DE DECISIÓN
# ============================================================

print("1) VARIABLES DE DECISIÓN")
print("xij = unidades del producto j producidas en la planta i")

for planta in plantas:
    print(f"y{planta} = 1 si se abre Planta {planta}")

print()

# ============================================================
# 4) FUNCIÓN OBJETIVO
# ============================================================

print("2) FUNCIÓN OBJETIVO")

terminos_utilidad = [
    f"{productos[producto]['utilidad']}x{planta}{producto}"
    for planta in plantas
    for producto in productos
]

terminos_costos = [
    f"{plantas[planta]['costo_fijo']}y{planta}"
    for planta in plantas
]

print("Max Z = " + " + ".join(terminos_utilidad) + " - " + " - ".join(terminos_costos))
print()

# ============================================================
# 5) RESTRICCIONES
# ============================================================

print("3) RESTRICCIONES")

for recurso, datos_recurso in recursos.items():
    print(f"{recurso}:")
    expresion = " + ".join(
        f"{productos[producto]['recursos'][recurso]}x{planta}{producto}"
        for planta in plantas
        for producto in productos
    )
    print(f"{expresion} <= {datos_recurso['disponible']}")
    print()

print("Capacidad por planta:")
for planta, datos in plantas.items():
    expresion = " + ".join(
        f"x{planta}{producto}"
        for producto in productos
    )
    print(f"{expresion} <= {datos['capacidad']}y{planta}")

print()

print("Demanda mínima:")
for producto, datos in productos.items():
    expresion = " + ".join(
        f"x{planta}{producto}"
        for planta in plantas
    )
    print(f"{expresion} >= {datos['demanda_minima']}")

print()

print("No negatividad y binarias")
print("xij >= 0, yi = 0 o 1")
print()

# ============================================================
# 6) SOLUCIÓN
# ============================================================

print("4) SOLUCIÓN")
estado = LpStatus[modelo.status]
print("Estado:", estado)
print()

if estado == "Optimal":

    print("Variables óptimas:")
    for planta in plantas:
        for producto in productos:
            print(f"x{planta}{producto} = {x[planta, producto].varValue}")

    print()

    for planta, datos in plantas.items():
        print(f"y{planta} ({datos['nombre']}) = {y[planta].varValue}")

    print()

    print("Ganancia máxima:")
    print(f"Z = {value(modelo.objective)}")
    print()

    print("Uso de recursos:")
    for recurso, datos_recurso in recursos.items():
        uso = sum(
            productos[producto]["recursos"][recurso] * x[planta, producto].varValue
            for planta in plantas
            for producto in productos
        )
        print(f"{recurso} usada = {uso} de {datos_recurso['disponible']}")

    print()

    print("Producción total:")
    for producto in productos:
        produccion = sum(
            x[planta, producto].varValue
            for planta in plantas
        )
        print(f"Producto {producto} = {produccion}")

    print()

    print("Capacidad utilizada:")
    for planta, datos in plantas.items():
        uso = sum(
            x[planta, producto].varValue
            for producto in productos
        )
        print(f"{datos['nombre']} = {uso} de {datos['capacidad']}")

    print()

    print("Interpretación:")
    for planta, datos in plantas.items():
        if y[planta].varValue == 1:
            print(f"Se abre {datos['nombre']}")
        else:
            print(f"No se abre {datos['nombre']}")

else:
    print("El modelo no tiene una solución factible.")
    print("No se deben interpretar los valores de las variables como solución óptima.")
    print()
    print("Posible causa:")
    print("Con los recursos disponibles no se puede cumplir simultáneamente")
    print("la demanda mínima de A y B junto con las demás restricciones.")

1) VARIABLES DE DECISIÓN
xij = unidades del producto j producidas en la planta i
y1 = 1 si se abre Planta 1
y2 = 1 si se abre Planta 2

2) FUNCIÓN OBJETIVO
Max Z = 30x1A + 40x1B + 30x2A + 40x2B - 600y1 - 500y2

3) RESTRICCIONES
Mano de obra:
3x1A + 2x1B + 3x2A + 2x2B <= 200

Materia prima:
2x1A + 3x1B + 2x2A + 3x2B <= 150

Capacidad por planta:
x1A + x1B <= 80y1
x2A + x2B <= 100y2

Demanda mínima:
x1A + x2A >= 40
x1B + x2B >= 30

No negatividad y binarias
xij >= 0, yi = 0 o 1

4) SOLUCIÓN
Estado: Infeasible

El modelo no tiene una solución factible.
No se deben interpretar los valores de las variables como solución óptima.

Posible causa:
Con los recursos disponibles no se puede cumplir simultáneamente
la demanda mínima de A y B junto con las demás restricciones.


# Planificación de inventarios multiproducto y multi-período en farmacia

## Contexto real
Este problema se basa en la gestión de inventarios en una farmacia ubicada en Barranquilla, donde se deben tomar decisiones de abastecimiento a lo largo de varios períodos. El objetivo es garantizar la disponibilidad de medicamentos mientras se minimizan los costos asociados al inventario y a los pedidos. Este tipo de modelos es ampliamente utilizado en cadenas de suministro farmacéuticas, donde existen restricciones de almacenamiento, demanda variable y condiciones operativas específicas

## Datos del problema

### Productos:
Medicamentos tipo A (ej. antibióticos 1)  
Medicamentos tipo B (ej. antibióticos 2)

### Períodos:
t=1,2,3t = 1, 2, 3t=1,2,3

### Costos de compra (por unidad):
Producto A: 10, 12, 11  
Producto B: 8, 9, 10

### Costos de almacenamiento:
Producto A: 2 por unidad  
Producto B: 3 por unidad

### Demandas:
Producto A: 30, 40, 35  
Producto B: 20, 25, 30

### Capacidad:
Capacidad máxima de almacenamiento: 100 unidades por período

### Inventario inicial:
0 unidades para todos

## Supuestos
La demanda debe satisfacerse completamente en cada período.  
Los productos pueden almacenarse para períodos futuros.  
No hay pérdidas de inventario.  
Los costos son lineales.  
Los pedidos pueden realizarse en cada período.

## Nivel de complejidad
Este problema es de nivel alto, ya que integra decisiones interdependientes en múltiples períodos y múltiples productos dentro de un entorno real de cadena de suministro. Su estructura multi-período implica dependencias temporales entre decisiones de inventario y abastecimiento, lo que incrementa significativamente la complejidad del modelo y su resolución.

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL ENTERA MIXTA
# Planificación de inventarios multiproducto y multi-período
# Farmacia en Barranquilla
# Ejercicio #5 - avanzado
# ============================================================

from pulp import LpMinimize, LpProblem, LpVariable, LpStatus, value, LpBinary

# ============================================================
# 1) DATOS
# ============================================================

P = ["A", "B"]
T = ["1", "2", "3"]

c_compra = {
    ("A", "1"): 10, ("A", "2"): 12, ("A", "3"): 11,
    ("B", "1"): 8,  ("B", "2"): 9,  ("B", "3"): 10
}

c_inv = {
    "A": 2,
    "B": 3
}

demanda = {
    ("A", "1"): 30, ("A", "2"): 40, ("A", "3"): 35,
    ("B", "1"): 20, ("B", "2"): 25, ("B", "3"): 30
}

inventario_inicial = {
    "A": 0,
    "B": 0
}

capacidad_almacenamiento = 100
M = 200

# ============================================================
# 2) FUNCIONES PARA GENERAR SALIDA PROGRAMADA
# ============================================================

def mostrar(titulo, lineas):
    print(titulo)
    for linea in lineas:
        print(linea)
    print()

def sumatoria(indices, expresion):
    return "".join(f"Σ{indice}" for indice in indices) + f" {expresion}"

def generar_variables():
    variables = {
        "qpt": "cantidad comprada del producto p en el período t",
        "ipt": "inventario del producto p al final del período t",
        "ypt": "1 si se realiza pedido del producto p en el período t"
    }

    return [
        f"{variable} = {descripcion}"
        for variable, descripcion in variables.items()
    ]

def generar_funcion_objetivo():
    return [
        "Min Z = "
        + sumatoria(["p", "t"], "cpt·qpt")
        + " + "
        + sumatoria(["p", "t"], "hp·ipt")
    ]

def generar_restricciones():
    restricciones = {
        "Balance de inventario": [
            "i[p,t-1] + q[p,t] - d[p,t] = i[p,t]"
        ],
        "Capacidad de almacenamiento": [
            sumatoria(["p"], "i[p,t]") + f" <= {capacidad_almacenamiento}"
        ],
        "Activación de pedido": [
            "q[p,t] <= M·y[p,t]"
        ],
        "No negatividad y binarias": [
            "q[p,t] >= 0",
            "i[p,t] >= 0",
            "y[p,t] = 0 o 1"
        ]
    }

    lineas = []

    for titulo, contenido in restricciones.items():
        lineas.append(f"{titulo}:")
        lineas.extend(contenido)
        lineas.append("")

    return lineas

# ============================================================
# 3) MODELO
# ============================================================

modelo = LpProblem("Inventarios_Farmacia", LpMinimize)

q = {
    (p, t): LpVariable(f"q{p}{t}", lowBound=0)
    for p in P
    for t in T
}

i = {
    (p, t): LpVariable(f"i{p}{t}", lowBound=0)
    for p in P
    for t in T
}

y = {
    (p, t): LpVariable(f"y{p}{t}", cat=LpBinary)
    for p in P
    for t in T
}

# ============================================================
# 4) FUNCIÓN OBJETIVO CON SUMATORIAS REALES
# ============================================================

modelo += (
    sum(c_compra[p, t] * q[p, t] for p in P for t in T)
    +
    sum(c_inv[p] * i[p, t] for p in P for t in T)
), "Costo_Total"

# ============================================================
# 5) RESTRICCIONES CON SUMATORIAS REALES
# ============================================================

# Balance de inventario
for p in P:
    for t in T:
        if t == "1":
            modelo += (
                inventario_inicial[p] + q[p, t] - demanda[p, t] == i[p, t]
            ), f"Balance_{p}_t{t}"
        else:
            t_anterior = str(int(t) - 1)
            modelo += (
                i[p, t_anterior] + q[p, t] - demanda[p, t] == i[p, t]
            ), f"Balance_{p}_t{t}"

# Capacidad de almacenamiento
for t in T:
    modelo += (
        sum(i[p, t] for p in P) <= capacidad_almacenamiento
    ), f"Capacidad_t{t}"

# Activación de pedido
for p in P:
    for t in T:
        modelo += (
            q[p, t] <= M * y[p, t]
        ), f"Activacion_{p}_t{t}"

# ============================================================
# 6) SOLUCIÓN
# ============================================================

modelo.solve()
estado = LpStatus[modelo.status]

# ============================================================
# 7) OUTPUT PROGRAMADO
# ============================================================

mostrar("1) VARIABLES DE DECISIÓN", generar_variables())

mostrar("2) FUNCIÓN OBJETIVO", generar_funcion_objetivo())

mostrar("3) RESTRICCIONES", generar_restricciones())

print("4) SOLUCIÓN")
print("Estado:", estado)
print()

if estado == "Optimal":

    print("Compras óptimas:")
    for p in P:
        for t in T:
            print(f"q{p}{t} = {q[p, t].varValue}")
    print()

    print("Inventarios óptimos:")
    for p in P:
        for t in T:
            print(f"i{p}{t} = {i[p, t].varValue}")
    print()

    print("Activación de pedidos:")
    for p in P:
        for t in T:
            print(f"y{p}{t} = {y[p, t].varValue}")
    print()

    print("Costo mínimo total:")
    print("Z =", value(modelo.objective))
    print()

    print("Verificación de capacidad de almacenamiento:")
    for t in T:
        uso = sum(i[p, t].varValue for p in P)
        print(f"t{t} = {uso} de {capacidad_almacenamiento}")

else:
    print("El modelo no tiene solución óptima.")

1) VARIABLES DE DECISIÓN
qpt = cantidad comprada del producto p en el período t
ipt = inventario del producto p al final del período t
ypt = 1 si se realiza pedido del producto p en el período t

2) FUNCIÓN OBJETIVO
Min Z = ΣpΣt cpt·qpt + ΣpΣt hp·ipt

3) RESTRICCIONES
Balance de inventario:
i[p,t-1] + q[p,t] - d[p,t] = i[p,t]

Capacidad de almacenamiento:
Σp i[p,t] <= 100

Activación de pedido:
q[p,t] <= M·y[p,t]

No negatividad y binarias:
q[p,t] >= 0
i[p,t] >= 0
y[p,t] = 0 o 1


4) SOLUCIÓN
Estado: Optimal

Compras óptimas:
qA1 = 70.0
qA2 = 0.0
qA3 = 35.0
qB1 = 20.0
qB2 = 25.0
qB3 = 30.0

Inventarios óptimos:
iA1 = 40.0
iA2 = 0.0
iA3 = 0.0
iB1 = 0.0
iB2 = 0.0
iB3 = 0.0

Activación de pedidos:
yA1 = 1.0
yA2 = 1.0
yA3 = 1.0
yB1 = 1.0
yB2 = 1.0
yB3 = 1.0

Costo mínimo total:
Z = 1850.0

Verificación de capacidad de almacenamiento:
t1 = 40.0 de 100
t2 = 0.0 de 100
t3 = 0.0 de 100


Una empresa manufacturera produce dos productos, P1 y P2, los cuales deben ser distribuidos a dos bodegas, B1 y B2, durante tres períodos de tiempo t = 1, 2 y 3. En cada período, la empresa decide cuánto producir de cada producto y cómo distribuirlo a las bodegas para satisfacer la demanda. Los productos pueden almacenarse en las bodegas, generando costos de inventario, y además producir un producto en un período implica una decisión discreta de activación.

Los costos de producción son los siguientes: para el producto P1, el costo es 5 en el período 1, 6 en el período 2 y 5 en el período 3. Para el producto P2, el costo es 4 en el período 1, 5 en el período 2 y 6 en el período 3. Los costos de almacenamiento para ambos productos son de 1 por unidad en la bodega B1 y de 2 por unidad en la bodega B2 en todos los períodos. Los costos de transporte son, para el producto P1, 3 por unidad hacia la bodega B1 y 4 hacia la bodega B2 en todos los períodos; y para el producto P2, 2 por unidad hacia la bodega B1 y 3 hacia la bodega B2 en todos los períodos.

La demanda que debe satisfacerse es la siguiente: para el producto P1, en la bodega B1 es de 20, 15 y 10 unidades en los períodos 1, 2 y 3 respectivamente, y en la bodega B2 es de 10, 15 y 20 unidades en los mismos períodos. Para el producto P2, en la bodega B1 es de 10, 20 y 15 unidades en los períodos 1, 2 y 3 respectivamente, y en la bodega B2 es de 15, 10 y 15 unidades en los mismos períodos. La capacidad máxima de producción en cada período es de 100 unidades, y el inventario inicial en todas las bodegas para ambos productos es igual a cero.

El objetivo es determinar cuánto producir de cada producto en cada período, cuánto enviar a cada bodega y cuánto inventario mantener, así como decidir si se activa o no la producción de cada producto en cada período, de manera que se minimice el costo total compuesto por los costos de producción, almacenamiento y transporte, cumpliendo con la demanda en cada período, respetando la capacidad de producción y garantizando el balance de inventarios a lo largo del tiempo.

In [ ]:
# ============================================================
# PROGRAMACIÓN LINEAL ENTERA MIXTA
# Ejercicio #6 - avanzado
# ============================================================

from pulp import LpMinimize, LpProblem, LpVariable, LpStatus, value, LpBinary

# ============================================================
# DATOS
# ============================================================

P = ["1", "2"]
B = ["1", "2"]
T = ["1", "2", "3"]

capacidad = 100
M = 100

c_prod = {
    ("1", "1"): 5, ("1", "2"): 6, ("1", "3"): 5,
    ("2", "1"): 4, ("2", "2"): 5, ("2", "3"): 6
}

c_env = {
    ("1", "1"): 3, ("1", "2"): 4,
    ("2", "1"): 2, ("2", "2"): 3
}

c_inv = {
    ("1", "1"): 1, ("1", "2"): 2,
    ("2", "1"): 1, ("2", "2"): 2
}

dem = {
    ("1", "1", "1"): 20, ("1", "1", "2"): 15, ("1", "1", "3"): 10,
    ("1", "2", "1"): 10, ("1", "2", "2"): 15, ("1", "2", "3"): 20,
    ("2", "1", "1"): 10, ("2", "1", "2"): 20, ("2", "1", "3"): 15,
    ("2", "2", "1"): 15, ("2", "2", "2"): 10, ("2", "2", "3"): 15
}

# ============================================================
# FUNCIONES PARA OUTPUT PROGRAMADO
# ============================================================

def mostrar_bloque(titulo, lineas):
    print(titulo)
    for linea in lineas:
        print(linea)
    print()

def mostrar_valores(titulo, variables, indices):
    print(titulo)
    for indice in indices:
        nombre = "".join(indice)
        print(f"{nombre} = {variables[indice].varValue}")
    print()

def sigma(indices, expresion):
    return "".join(f"Σ{indice}" for indice in indices) + f" {expresion}"

def generar_variables_decision():
    variables = {
        "xpt": "producción",
        "epbt": "envío",
        "ipbt": "inventario",
        "ypt": "activación"
    }

    return [f"{var} = {desc}" for var, desc in variables.items()]

def generar_funcion_objetivo():
    partes = [
        sigma(["p", "t"], "cpt·xpt"),
        sigma(["p", "b", "t"], "fpb·epbt"),
        sigma(["p", "b", "t"], "hpb·ipbt")
    ]

    return ["Min Z = " + " + ".join(partes)]

def generar_restricciones():
    restricciones = {
        "Capacidad": [
            sigma(["p"], "xpt") + f" <= {capacidad}"
        ],
        "Activación": [
            "xpt <= M·ypt"
        ],
        "Producción-envío": [
            "xpt = " + sigma(["b"], "epbt")
        ],
        "Inventario": [
            "ipb(t-1) + epbt - dpbt = ipbt"
        ],
        "No negatividad": [
            "xpt >= 0",
            "epbt >= 0",
            "ipbt >= 0",
            "ypt binaria"
        ]
    }

    lineas = []

    for nombre, contenido in restricciones.items():
        lineas.append(f"{nombre}:")
        lineas.extend(contenido)
        lineas.append("")

    return lineas

def generar_interpretacion():
    return [
        "El modelo determina cuánto producir de cada producto en cada período,",
        "cuánto enviar a cada bodega y cuánto inventario mantener al final",
        "de cada período.",
        "",
        "La solución minimiza el costo total considerando producción,",
        "transporte e inventario, respetando la capacidad y satisfaciendo",
        "la demanda de cada producto en cada bodega."
    ]

# ============================================================
# MODELO
# ============================================================

modelo = LpProblem("Modelo", LpMinimize)

x = {
    (p, t): LpVariable(f"x{p}{t}", lowBound=0)
    for p in P
    for t in T
}

e = {
    (p, b, t): LpVariable(f"e{p}{b}{t}", lowBound=0)
    for p in P
    for b in B
    for t in T
}

i = {
    (p, b, t): LpVariable(f"i{p}{b}{t}", lowBound=0)
    for p in P
    for b in B
    for t in T
}

y = {
    (p, t): LpVariable(f"y{p}{t}", cat=LpBinary)
    for p in P
    for t in T
}

# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================

modelo += (
    sum(c_prod[p, t] * x[p, t] for p in P for t in T)
    +
    sum(c_env[p, b] * e[p, b, t] for p in P for b in B for t in T)
    +
    sum(c_inv[p, b] * i[p, b, t] for p in P for b in B for t in T)
), "Costo_Total"

# ============================================================
# RESTRICCIONES
# ============================================================

for t in T:
    modelo += (
        sum(x[p, t] for p in P) <= capacidad
    ), f"Capacidad_t{t}"

for p in P:
    for t in T:
        modelo += (
            x[p, t] <= M * y[p, t]
        ), f"Activacion_{p}_{t}"

for p in P:
    for t in T:
        modelo += (
            x[p, t] == sum(e[p, b, t] for b in B)
        ), f"Balance_Produccion_{p}_{t}"

for p in P:
    for b in B:
        for t in T:
            if t == "1":
                modelo += (
                    e[p, b, t] - dem[p, b, t] == i[p, b, t]
                ), f"Inventario_{p}_{b}_{t}"
            else:
                t_anterior = str(int(t) - 1)
                modelo += (
                    i[p, b, t_anterior]
                    + e[p, b, t]
                    - dem[p, b, t]
                    == i[p, b, t]
                ), f"Inventario_{p}_{b}_{t}"

# ============================================================
# SOLUCIÓN
# ============================================================

modelo.solve()
estado = LpStatus[modelo.status]

# ============================================================
# OUTPUT PROGRAMADO
# ============================================================

mostrar_bloque("1) VARIABLES DE DECISIÓN", generar_variables_decision())
mostrar_bloque("2) FUNCIÓN OBJETIVO", generar_funcion_objetivo())
mostrar_bloque("3) RESTRICCIONES", generar_restricciones())

print("4) SOLUCIÓN")
print("Estado:", estado)
print()

if estado == "Optimal":

    mostrar_valores(
        "Producción:",
        x,
        [(p, t) for p in P for t in T]
    )

    mostrar_valores(
        "Envíos:",
        e,
        [(p, b, t) for p in P for b in B for t in T]
    )

    mostrar_valores(
        "Inventarios:",
        i,
        [(p, b, t) for p in P for b in B for t in T]
    )

    mostrar_valores(
        "Activación:",
        y,
        [(p, t) for p in P for t in T]
    )

    print("Costo mínimo total:")
    print("Z =", value(modelo.objective))
    print()

    mostrar_bloque(
        "Verificación de capacidad:",
        [
            f"t{t} = {sum(x[p, t].varValue for p in P)} de {capacidad}"
            for t in T
        ]
    )

    mostrar_bloque(
        "Demanda satisfecha por período:",
        [
            f"P{p}-B{b}: " + ", ".join(
                f"t{t}="
                + str(
                    e[p, b, t].varValue
                    if t == "1"
                    else i[p, b, str(int(t) - 1)].varValue + e[p, b, t].varValue
                )
                for t in T
            )
            for p in P
            for b in B
        ]
    )

    mostrar_bloque("Interpretación:", generar_interpretacion())

else:
    print("El modelo no tiene solución óptima.")
    print("Estado encontrado:", estado)

1) VARIABLES DE DECISIÓN
xpt = producción
epbt = envío
ipbt = inventario
ypt = activación

2) FUNCIÓN OBJETIVO
Min Z = ΣpΣt cpt·xpt + ΣpΣbΣt fpb·epbt + ΣpΣbΣt hpb·ipbt

3) RESTRICCIONES
Capacidad:
Σp xpt <= 100

Activación:
xpt <= M·ypt

Producción-envío:
xpt = Σb epbt

Inventario:
ipb(t-1) + epbt - dpbt = ipbt

No negatividad:
xpt >= 0
epbt >= 0
ipbt >= 0
ypt binaria


4) SOLUCIÓN
Estado: Optimal

Producción:
11 = 45.0
12 = 15.0
13 = 30.0
21 = 45.0
22 = 10.0
23 = 30.0

Envíos:
111 = 35.0
112 = 0.0
113 = 10.0
121 = 10.0
122 = 15.0
123 = 20.0
211 = 30.0
212 = 0.0
213 = 15.0
221 = 15.0
222 = 10.0
223 = 15.0

Inventarios:
111 = 15.0
112 = 0.0
113 = 0.0
121 = 0.0
122 = 0.0
123 = 0.0
211 = 20.0
212 = 0.0
213 = 0.0
221 = 0.0
222 = 0.0
223 = 0.0

Activación:
11 = 1.0
12 = 1.0
13 = 1.0
21 = 1.0
22 = 1.0
23 = 1.0

Costo mínimo total:
Z = 1435.0

Verificación de capacidad:
t1 = 90.0 de 100
t2 = 25.0 de 100
t3 = 60.0 de 100

Demanda satisfecha por período:
P1-B1: t1=35.0, t2=15.0, t3=10.0
P1-B2: 